# aw_09_b6 — Stage B6: PlayWorld GRPO from the Phase-2 champion (Track B, §5.1)

**Parent = B4v2** (`20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2`,
sha256:d4fcacdd...; §6 Phase-2 champion — B5 DPO was ~null vs B4v2, so GRPO also
starts from the SFT champion). This gives the clean triangle from ONE parent:
B4v2 (SFT) vs B5 (offline DPO) vs B6 (online GRPO), all on the frozen suites.

Rewards: `verifier_reward_function(default_playworld_verifier)` — the SAME
verifier as eval, applied online to rollouts (episode-level, no reward model).
Prompts: canonical frozen `train/v1/playworld_prompts.jsonl` (v1.3 rule).

Cell order: fetch parent → `a_b6_data` → `b_b6_train` → `c_b6_eval` →
`x09j_run_audit` → `f_b6_analysis` (B6 vs B4v2, B6 vs B5 = headline).


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt
# v0.6.12 amend (2026-08-16 ABI incident): NEVER blind-install the vLLM lock.
# Install it ONLY if it carries an exact ABI-verified pin (written after
# scripts/x18_vllm_compat_probe.py PASS on a fresh G4 runtime). An unpinned
# lock is skipped -> run training with
#   --override training.extra.use_vllm=false   (HF-generate fallback).
!grep -q '^vllm==' requirements/vllm.lock.txt \
  && pip install -r requirements/vllm.lock.txt \
  || echo '[header] vllm.lock.txt has no exact pin - SKIPPING vLLM install (run x18 probe first; use --override training.extra.use_vllm=false)'


Cloning into 'axiom-world'...
remote: Enumerating objects: 748, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 748 (delta 179), reused 187 (delta 84), pack-reused 451 (from 1)
Receiving objects: 100% (748/748), 329.64 KiB | 1.82 MiB/s, done.
Resolving deltas: 100% (411/411), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 66.8 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... don

In [ ]:
# @title fetch parent — B4v2 champion adapter + lineage sha
B4V2_RUN_ID = "20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2"

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_RUN_ID}
b4v2_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

import json
b4v2_sha = json.load(open(f"runs/{B4V2_RUN_ID}/artifacts/lineage.json"))["output_adapter_sha256"]
print(b4v2_dir, b4v2_sha)


runs/20260814-023603--b4v2-playworld-sft-from-p1--s42--c56ed2/artifacts/final_adapter sha256:d4fcacddf21f758cdab904845ebdfee1eefde309c0edb6205bac64d5f07c76c8


In [ ]:
# @title a_b6_data — frozen prompts + suites (sha-pinned, v1.3 rule)
!python scripts/fetch_dataset.py \
  --repo m97j/aw-playworld --path train/v1/playworld_prompts.jsonl \
  --output data/train/playworld_prompts.jsonl --force \
  --expected-sha256 2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6

!python scripts/build_eval_suites.py --episodes-per-suite 300
# freeze_fingerprint MUST equal sha256:3cdcbc30... — abort otherwise


playworld_prompts.jsonl: 100% 4.60M/4.60M [00:00<00:00, 34.8MB/s]
fetched dataset: hf://m97j/aw-playworld/train/v1/playworld_prompts.jsonl
revision: main
materialized: data/train/playworld_prompts.jsonl
dataset sha256: 2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6
DATASET_PATH=data/train/playworld_prompts.jsonl
DATASET_SHA256=2e7d02603c47784328ee82bba8abb6ed8b9e32175567b72b7a68969a0ae361e6
eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training load

In [ ]:
# @title x17_scenario_audit — GRPO scenario-transport audit (MUST PASS before training)
# Proves the v0.6.12 fix on the ACTUAL frozen prompts file:
#   - legacy dict column: reports how many rows Arrow mutates/None-injects
#     (evidence of the aborted 20260815-020335 run's root cause)
#   - fixed scenario_json string column: 0 round-trip mismatches, 0
#     validation failures required
#   - reward parity: oracle completions scored via the fixed path must equal
#     direct (no-Arrow) scoring sample-by-sample
# Exit code != 0 -> DO NOT TRAIN; paste runs/x17_grpo_scenario_audit.json.
!python scripts/x17_grpo_scenario_audit.py \
  --prompts data/train/playworld_prompts.jsonl \
  --out runs/x17_grpo_scenario_audit.json


{
  "prompts": "data/train/playworld_prompts.jsonl",
  "n_rows": 2000,
  "legacy_dict_column": {
    "rows_mutated_by_arrow": 2000,
    "validation_failures": 2000
  },
  "fixed_json_string_column": {
    "roundtrip_mismatches": 0,
    "validation_failures": 0
  },
  "reward_parity": {
    "n_scored": 64,
    "n_equal": 64,
    "direct_none": 0,
    "fixed_none": 0,
    "fixed_mean_reward": 1.0
  },
  "verdict": "PASS"
}


In [ ]:
# @title b_b6_train — GRPO from the B4v2 parent (online verifier reward)
# Watch reward/mean in the logs: it should start near B4v2's train-distribution
# pass level and climb. If reward flatlines at 0 or 1, stop and report.
!python scripts/run_experiment.py \
  --config configs/experiments/b6_playworld_grpo.yaml \
  --parent-adapter-dir {b4v2_dir} \
  --override lineage.parent_run_id={B4V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4v2_sha} \
  --override data.source.local_path=data/train/playworld_prompts.jsonl \
  --override training.extra.use_vllm=false \
  --hf-sync-repo m97j/aw-runs-b6


Streaming output truncated to the last 5000 lines.
  Preparing   ████████████████████  4 / 4 ✓
  Uploading   ████████████████████  -
  Committing  ████████████████████  4 / 4 ✓
No files have been modified since last commit. Skipping to prevent empty commit.
{'loss': '0.04017', 'grad_norm': '1.396', 'learning_rate': '5.881e-07', 'num_tokens': '2.767e+06', 'completions/mean_length': '95.05', 'completions/min_length': '83.5', 'completions/max_length': '107.3', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '95.05', 'completions/min_terminated_length': '83.5', 'completions/max_terminated_length': '107.3', 'rewards/verifier_reward_hybrid/mean': '0.225', 'rewards/verifier_reward_hybrid/std': '0.2205', 'reward': '0.225', 'reward_std': '0.2205', 'frac_reward_zero_std': '0.5', 'entropy': '0.01364', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '5.789', 'epoch': 

In [ ]:
# @title b_b6_resume — restart an interrupted b_b6_train from the hub checkpoint
# Run AFTER a runtime disconnect: rerun cells 1-3 first (header, fetch parent,
# a_b6_data), then this cell INSTEAD of b_b6_train. --resume-from hub pulls the
# newest checkpoint-N from m97j/aw-runs-b6 (pushed by HFCheckpointSync every
# save_steps=50) and continues via trainer_state; if no checkpoint exists yet
# (interrupted before the first save) it prints "no hub checkpoint found;
# starting fresh" and restarts cleanly. Either way the finished run gets a NEW
# run_id — take it from the "run_id: ..." log line for c_b6_eval.
!python scripts/run_experiment.py \
  --config configs/experiments/b6_playworld_grpo.yaml \
  --parent-adapter-dir {b4v2_dir} \
  --override lineage.parent_run_id={B4V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4v2_sha} \
  --override data.source.local_path=data/train/playworld_prompts.jsonl \
  --hf-sync-repo m97j/aw-runs-b6 \
  --resume-from hub


In [ ]:
# @title c_b6_eval — B6 adapter on the frozen suites (canonical profile)
B6_RUN_ID = "20260815-150717--b6-playworld-grpo--s42--6e6c45"  # <- from b_b6_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6_RUN_ID}
b6_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b6_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b6


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 342.77it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
generate(batched): 100% 3/3 [00:31<00:00, 10.62s/it]
eval_adversarial: pass_rate={'mean': 0.75, 'c

In [ ]:
# @title x09j_run_audit — termination regression check on the B6 eval (CPU)
B6_EVAL = "20260816-004824--eval-playworld--s42--274abd"  # <- eval run id from c_b6_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B6_EVAL} --out runs/x09_run_audit_b6.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.31k [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  82% 3.31k/4.03k [00:00<00:00, 15.3kB/s]
Reconstructing (incomplete total...):   1% 4.03k/392k [00:00<00:25, 15.3kB/s] 
Reconstructing (incomplete total...):   0% 4.03k/853k [00:00<00:55, 15.3kB/s]
Reconstructing (incomplete total...):  30% 392k/1.30M [00:00<00:00, 1.33MB/s]
Reconstructing (incomplete total...): 100% 1.30M/1.30M [00:00<00:00, 1.33MB/s]

Fetching 8 files:  12% 1/8 [00:00<00:03,  2.31it/s]
Reconstructing (incomplete total...):  74% 1.30M/1.77M [00:00<00:00, 1.33MB/s]
Reconstructing (incomplete total...):  79% 1.77M/2.23M [00:00<00:00, 5.30MB/s]

Fetching 8 files: 100% 8/8 [00:00<00:00, 13.69it/s]
Download complete: 100% 2.23M/2.23M [00:00<00:00, 5.30MB/s]
Reconstruction complete: 100% 2.23M/2.23M [00:00<00:00, 5.30MB/s]             ev

In [ ]:
# @title x19_b6_regression_diag
!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --kind eval \
  --run-id 20260816-004824--eval-playworld--s42--274abd
!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --kind eval \
  --run-id 20260814-032546--eval-playworld--s42--7308ee

!python scripts/x19_b6_regression_diag.py \
  --run-a runs/20260816-004824--eval-playworld--s42--274abd \
  --run-b runs/20260814-032546--eval-playworld--s42--7308ee \
  --label-a b6-grpo --label-b b4v2-sft \
  --out runs/x19_b6_regression_diag.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/388k [00:00<?, ?B/s]          
Reconstructing (incomplete total...):  46% 388k/848k [00:00<00:00, 1.22MB/s]
Reconstructing (incomplete total...):  30% 388k/1.31M [00:00<00:00, 1.22MB/s]
Reconstructing (incomplete total...):  22% 388k/1.76M [00:00<00:01, 1.22MB/s]
Reconstructing (incomplete total...):  17% 388k/2.22M [00:00<00:01, 1.22MB/s]
Reconstructing (incomplete total...): 100% 2.22M/2.23M [00:00<00:00, 1.22MB/s]

Fetching 19 files:   5% 1/19 [00:00<00:06,  2.70it/s]
Reconstructing (incomplete total...): 100% 2.23M/2.23M [00:00<00:00, 1.22MB/s]
Reconstructing (incomplete total...): 100% 2.23M/2.24M [00:00<00:00, 1.22MB/s]
Reconstructing (incomplete total...): 100% 2.24M/2.24M [00:00<00:00, 1.22MB/s]

Fetching 19 files:  47% 9/19 [00:00<00:00, 17.43it/s]
Reconstructing (incomplete total...):  85% 2.24M/2.63M [00

In [ ]:
# @title f_b6_analysis — B6 vs B4v2 (RL gain) and B6 vs B5 (online vs offline)
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"
B5_EVAL = "20260814-124224--eval-playworld--s42--f77cd8"

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b5 --run-id {B5_EVAL} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{B6_EVAL} --label-a b6-grpo --run-b runs/{B4V2_EVAL} --label-b b4v2-sft \
  --output runs/{B6_EVAL}/analysis_b6_vs_b4v2.json --hf-sync-repo m97j/aw-runs-b6

!python scripts/run_analysis.py \
  --run-a runs/{B6_EVAL} --label-a b6-grpo --run-b runs/{B5_EVAL} --label-b b5-dpo \
  --output runs/{B6_EVAL}/analysis_b6_vs_b5.json --hf-sync-repo m97j/aw-runs-b6


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/577 [00:00<?, ?B/s]           
Reconstructing (incomplete total...):   0% 577/434k [00:00<03:45, 1.92kB/s]
Reconstructing (incomplete total...):   0% 577/821k [00:00<07:06, 1.92kB/s]
Reconstructing (incomplete total...):   0% 577/827k [00:00<07:10, 1.92kB/s]
Reconstructing (incomplete total...):   1% 6.70k/1.28M [00:00<10:59, 1.92kB/s]
Reconstructing (incomplete total...):   0% 6.70k/1.73M [00:00<14:54, 1.92kB/s]
Reconstructing (incomplete total...):   0% 6.70k/2.17M [00:00<18:46, 1.92kB/s]
Reconstructing (incomplete total...):  38% 827k/2.18M [00:00<11:42, 1.92kB/s] 

Fetching 19 files:   5% 1/19 [00:00<00:06,  2.75it/s]
Reconstructing (incomplete total...): 100% 2.18M/2.18M [00:00<00:01, 1.92kB/s]

Fetching 19 files:  47% 9/19 [00:00<00:00, 17.72it/s]
Reconstructing (incomplete total...):  83% 2.18M/2.63M [00:00<

## Stage checklist
- [x] a_b6_data: prompts sha 2e7d0260 verified; suites freeze 3cdcbc30 verified
- [x] b_b6_train: reward/mean trajectory recorded (start≈parent level, climbing);
      lineage verified (parent sha d4fcacdd)
- [x] c_b6_eval per-suite pass_rate; x09j truncation/runaway ≈ 0
- [x] f_b6_analysis: B6 vs B4v2 and **B6 vs B5** recorded
- [x] §6 final Track-B champion (B4v2/B5/B6) → seeds (aw_11) + ablations (aw_10) + report

- [x] x17 scenario-transport audit verdict PASS (fixed path clean, reward parity full)
- [x] train log shows nonzero reward/reward_std within the first logged steps
- [x] no RewardHealthError raised (guard: >50% excluded after 256 completions aborts)
- [x] x18 vLLM ABI probe PASS on fresh G4 (baseline torch 2.11.0+cu128 untouched) and exact pin committed to requirements/vllm.lock.txt


---
# B6-R — rerun with PASS-GATED reward (§7.4 amendment, post-x19)

**Why this section exists.** B6 above (run
`20260815-150717--b6-playworld-grpo--s42--6e6c45`, aggregate reward) trained
cleanly but REGRESSED pass_rate vs its own parent on every frozen suite
(ID −14.0pp, template −14.0pp, comp −8.7pp, adversarial −10.3pp; all
significant) while mean_score stayed ~flat. x19
(`runs/x19_b6_regression_diag.json`) localized the mechanism: 40–50% of
parent-pass→B6-fail flips carry B6 scores ≥ 0.8 with exactly one
`required_component_failed`, and predictions grew +27–37 chars — the aggregate
reward paid near-misses almost as much as passes (**objective mismatch**, not a
training bug). Style-overfit (shorter plans) was REJECTED by the same data.

**Single-variable change** (config `b6r_playworld_grpo_gated.yaml`):
`training.extra.reward_mode: pass_gated` — passed → 0.5 + 0.5·score,
failed → 0.1·score (worst pass 0.5 > best fail 0.1). Parent, data, budget,
sampling, seed identical to B6 → B6-R-vs-B6 is causal for the reward shape.

**Pre-registered decision rule (§6)**:
- B6-R ≥ B4v2 on all suites (n.s. or better) → reward alignment fixed the regression.
- Any suite significantly ABOVE B4v2 → online-RL headroom exists (RQ2 reversal).
- Still significantly below → SFT champion stands; RL did not beat it in 2 attempts.

**Prerequisites**: cells 1–4 above (common header → fetch parent → a_b6_data →
x17 gate) — shared with B6, run them first on a fresh runtime.
**Note on sync repo**: this campaign syncs to `m97j/aw-runs-b6` (same repo as
B6; runs are distinguished by run_id / experiment_name `b6r-playworld-grpo-gated`).

In [ ]:
# @title b_b6r_vllm_server — start isolated rollout server (background) BEFORE b_b6r_train
import subprocess, time, urllib.request

subprocess.Popen(
    ["bash", "scripts/launch_trl_vllm_server.sh", "Qwen/Qwen3-8B", "8000"],
    stdout=open("/content/vllm_server.log", "w"), stderr=subprocess.STDOUT)

# Wait until model loads (several minutes based on G1 smoke)
for i in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print("vLLM server ready"); break
    except Exception:
        time.sleep(10)
else:
    raise RuntimeError("server not ready — check /content/vllm_server.log")
!tail -5 /content/vllm_server.log


vLLM server ready
Capturing CUDA graphs (FULL): 100%|██████████| 51/51 [00:39<00:00,  1.28it/s]
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:39542 - "GET /health HTTP/1.1" 307 Temporary Redirect
INFO:     127.0.0.1:39550 - "GET /health/ HTTP/1.1" 200 OK


In [ ]:
# @title b_b6r_train — GRPO from B4v2 with reward_mode=pass_gated
# Reward SCALE differs from B6 by design — do NOT compare absolute reward/mean
# with the B6 run. Healthy signs: reward/mean starts ≈0.05–0.15
# (≈0.5×pass-rate + small fail credit) and CLIMBS; frac_reward_zero_std
# declines; entropy stays > ~0.005; clipped_ratio → 0; no RewardHealthError.
# Take the "run_id: ..." log line for c_b6r_eval.
!python scripts/run_experiment.py \
  --config configs/experiments/b6r_playworld_grpo_gated.yaml \
  --parent-adapter-dir {b4v2_dir} \
  --override lineage.parent_run_id={B4V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4v2_sha} \
  --override data.source.local_path=data/train/playworld_prompts.jsonl \
  --override training.save_steps=250 \
  --hf-sync-repo m97j/aw-runs-b6


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260816-155851--b6r-playworld-grpo-gated--s42--de963e
config.json: 100% 729/729 [00:00<00:00, 5.79MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 4.01MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 59.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 191MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 78.7MB/s]
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 146MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.95G [00

In [ ]:
# @title b_b6r_resume — ONLY after a runtime disconnect during b_b6r_train
# Rerun cells 1–4 first, then this cell INSTEAD of b_b6r_train. Pulls the
# newest checkpoint-N from m97j/aw-runs-b6 (pushed every save_steps=50); starts
# fresh if none. The finished run gets a NEW run_id either way.
!python scripts/run_experiment.py \
  --config configs/experiments/b6r_playworld_grpo_gated.yaml \
  --parent-adapter-dir {b4v2_dir} \
  --override lineage.parent_run_id={B4V2_RUN_ID} \
  --override lineage.parent_adapter.sha256={b4v2_sha} \
  --override data.source.local_path=data/train/playworld_prompts.jsonl \
  --hf-sync-repo m97j/aw-runs-b6 \
  --resume-from hub

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260816-152110--b6r-playworld-grpo-gated--s42--06ffcb
config.json: 100% 729/729 [00:00<00:00, 5.11MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 7.29MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 32.3MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 2.57MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 20.4MB/s]
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 93.0MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.15G [

In [ ]:
# @title hf_prune_checkpoints

# Check for PRUNE determination with dry-run (quotation marks required — <root> must not be interpreted as a shell redirect)
!python scripts/hf_prune_checkpoints.py --repo m97j/aw-runs-b6 --also-incomplete "<root>"

# If the report changes to "PRUNE <root>: ... dropping 39 ckpts / 588.0GB ... keeping [2000]":
!python scripts/hf_prune_checkpoints.py --repo m97j/aw-runs-b6 --also-incomplete "<root>" --execute

PRUNE              <root>: 40 ckpts / 595.5GB LFS; dropping 39 ckpts / 588.0GB; keeping steps [2000]

non-checkpoint LFS in repo: 0.0B
LFS blobs to PERMANENTLY delete: 280 files, ~588.0GB (execute=False)
NOTE: quota counts ALL revisions' blobs; this permanent deletion (rewrite_history=True) is the only path that actually frees space.
PRUNE              <root>: 40 ckpts / 595.5GB LFS; dropping 39 ckpts / 588.0GB; keeping steps [2000]

non-checkpoint LFS in repo: 0.0B
LFS blobs to PERMANENTLY delete: 280 files, ~588.0GB (execute=True)
NOTE: quota counts ALL revisions' blobs; this permanent deletion (rewrite_history=True) is the only path that actually frees space.
permanently deleted. Storage accounting may take a while to refresh.


In [ ]:
# @title x20_eval_identity_audit — did the "B6-R eval" score the same weights as the B6 eval?
B6R_EVAL = "20260816-121134--eval-playworld--s42--f1854f"   # suspect (B6r eval)
B6_EVAL  = "20260816-004824--eval-playworld--s42--274abd"   # reference (B6 eval)

!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6R_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6_EVAL} --kind eval

!python scripts/x20_eval_identity_audit.py \
  --run-a runs/{B6R_EVAL} \
  --run-b runs/{B6_EVAL} \
  --repo m97j/aw-runs-b6 --out runs/x20_eval_identity_audit.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/577 [00:00<?, ?B/s]           
Reconstructing (incomplete total...):   9% 577/6.70k [00:00<00:04, 1.32kB/s]

Fetching 19 files:   5% 1/19 [00:00<00:07,  2.27it/s]
Reconstructing (incomplete total...):  53% 6.70k/12.6k [00:00<00:04, 1.32kB/s]
Reconstructing (incomplete total...):   3% 12.6k/463k [00:00<05:41, 1.32kB/s] 
Reconstructing (incomplete total...):   1% 12.6k/922k [00:00<11:30, 1.32kB/s]
Reconstructing (incomplete total...):  33% 463k/1.39M [00:00<11:41, 1.32kB/s]
Reconstructing (incomplete total...):  78% 1.39M/1.77M [00:00<04:54, 1.32kB/s]
Reconstructing (incomplete total...):  62% 1.39M/2.24M [00:00<10:44, 1.32kB/s]

Fetching 19 files:  21% 4/19 [00:00<00:01,  8.94it/s]
Reconstructing (incomplete total...): 100% 2.24M/2.24M [00:00<00:00, 4.25MB/s]

Fetching 19 files:  47% 9/19 [00:00<00:00, 12.34it/s]
Re

In [ ]:
# @title b_b6r_upload_salvage — INCIDENT 2026-08-16: final sync hit HF storage quota
# B6-R training completed (2000/2000) but HFCheckpointSync's final commit failed
# with HF 400 "setup automatic credit recharge" (account storage quota — HF bills
# the LFS blobs of ALL revisions, so overwritten checkpoints kept charging).
# After freeing quota (scripts/hf_prune_checkpoints.py --also-incomplete "<root>"
# --execute), run this from the STILL-ALIVE training runtime.
#
# PATH CONTRACT (bug fixed 2026-08-16): fetch_run --kind adapter reads the repo
# ROOT artifacts/ — NOT {run_id}/artifacts/. An earlier salvage uploaded to the
# namespaced path, fetch_run then served the STALE B6-era root artifacts/, and
# the "B6-R eval" scored the wrong weights (caught by x20: all-zero deltas vs
# B6). Upload to the ROOT slot; lineage.json's run_id disambiguates the owner.
B6R_RUN_ID = "20260816-065258--b6r-playworld-grpo-gated--s42--dc7c8e"

import json
from huggingface_hub import HfApi

lineage = json.load(open(f"runs/{B6R_RUN_ID}/artifacts/lineage.json"))
assert lineage["run_id"] == B6R_RUN_ID, lineage["run_id"]  # never salvage the wrong run

api = HfApi()
api.upload_folder(
    repo_id="m97j/aw-runs-b6",
    folder_path=f"runs/{B6R_RUN_ID}/artifacts",
    path_in_repo="artifacts",   # ROOT slot — the path fetch_run actually reads
    commit_message=f"salvage: final artifacts for {B6R_RUN_ID} (quota incident 2026-08-16)",
)
print("root artifacts/ now holds:", lineage["run_id"])
print("adapter sha256:", lineage["output_adapter_sha256"])
# Verify on the hub before ending the runtime: artifacts/lineage.json run_id
# must equal B6R_RUN_ID. c_b6r_eval's fetch_run will re-verify the sha256.

FileNotFoundError: [Errno 2] No such file or directory: 'runs/20260816-065258--b6r-playworld-grpo-gated--s42--dc7c8e/artifacts/lineage.json'

In [ ]:
# @title c_b6r_eval — B6-R adapter on the frozen suites (canonical profile)
B6R_RUN_ID = "20260816-155851--b6r-playworld-grpo-gated--s42--de963e"  # <- from b_b6r_train "run_id: ..."

out = !python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6R_RUN_ID}
b6r_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {b6r_dir} \
  --max-new-tokens 1024 --batch-size 100 --hf-sync-repo m97j/aw-runs-b6

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 337.71it/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
generate(batched): 100% 3/3 [00:32<00:00, 10.95s/it]
eval_adversarial: pass_rate={'mean': 0.8, 'ci

In [ ]:
# @title x09jr_run_audit — termination regression check on the B6-R eval (CPU)
B6R_EVAL = "20260816-201107--eval-playworld--s42--8791aa"  # <- eval run id from c_b6r_eval

!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6R_EVAL} --kind eval
!python scripts/x09_termination_audit.py \
  --run-dirs runs/{B6R_EVAL} --out runs/x09_run_audit_b6r.json

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/577 [00:00<?, ?B/s]           

Fetching 8 files:  12% 1/8 [00:00<00:01,  5.27it/s]
Reconstructing (incomplete total...):  15% 577/3.94k [00:00<00:01, 2.97kB/s]
Reconstructing (incomplete total...):   1% 3.94k/467k [00:00<02:36, 2.97kB/s]
Reconstructing (incomplete total...):   0% 3.94k/928k [00:00<05:11, 2.97kB/s]
Reconstructing (incomplete total...):  67% 928k/1.38M [00:00<02:31, 2.97kB/s]
Reconstructing (incomplete total...):  53% 928k/1.77M [00:00<04:42, 2.97kB/s]
Reconstructing (incomplete total...):  53% 928k/1.77M [00:00<04:42, 2.97kB/s]
Fetching 8 files: 100% 8/8 [00:00<00:00, 28.40it/s]
Download complete: 100% 2.23M/2.23M [00:00<00:00, 2.96kB/s]
Reconstruction complete: 100% 2.23M/2.23M [00:00<00:00, 2.97kB/s]            eval run materialized: 5 suite files, freeze_fingerprint=sha256:3cdcbc30c99e492c...
RUN_

In [ ]:
# @title x19_regression_diag — B6-R vs B6
!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --kind eval \
  --run-id 20260816-201107--eval-playworld--s42--8791aa            # B6R
!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --kind eval \
  --run-id 20260816-004824--eval-playworld--s42--274abd            # B6

!python scripts/x19_b6_regression_diag.py \
  --run-a runs/20260816-201107--eval-playworld--s42--8791aa \
  --run-b runs/20260816-004824--eval-playworld--s42--274abd \
  --label-a B6-R --label-b B6 \
  --out runs/x19_b6_regression_diag.json


eval run materialized: 5 suite files, freeze_fingerprint=sha256:3cdcbc30c99e492c...
RUN_DIR=runs/20260816-201107--eval-playworld--s42--8791aa
eval run materialized: 5 suite files, freeze_fingerprint=sha256:3cdcbc30c99e492c...
RUN_DIR=runs/20260816-004824--eval-playworld--s42--274abd
{
  "labels": {
    "a": "B6-R",
    "b": "B6"
  },
  "run_a": "runs/20260816-201107--eval-playworld--s42--8791aa",
  "run_b": "runs/20260816-004824--eval-playworld--s42--274abd",
  "interpretation_guide": {
    "aggregate_reward_hypothesis": "supported if parentpass_b6fail scores cluster in [0.4,0.8) \u2014 B6 keeps partial credit where parent passed",
    "style_overfit_hypothesis": "supported if prediction_len_delta is strongly negative and flips are uniform across score buckets"
  },
  "suites": {
    "eval_adversarial": {
      "episodes_paired": 300,
      "flip_matrix_parent_to_b6": {
        "fail->pass": 25,
        "pass->pass": 215,
        "fail->fail": 50,
        "pass->fail": 10
      },
    

In [ ]:
# @title f_b6r_analysis — B6-R vs B4v2 (headline), vs B6 (reward-shape effect), + x19 flips
B4V2_EVAL = "20260814-032546--eval-playworld--s42--7308ee"
B6_EVAL   = "20260816-004824--eval-playworld--s42--274abd"

!python scripts/fetch_run.py --repo m97j/aw-runs-b4 --run-id {B4V2_EVAL} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-b6 --run-id {B6_EVAL} --kind eval

# Headline: did pass-gating close the regression vs the SFT champion?
!python scripts/run_analysis.py \
  --run-a runs/{B6R_EVAL} --label-a b6r-grpo-gated --run-b runs/{B4V2_EVAL} --label-b b4v2-sft \
  --output runs/{B6R_EVAL}/analysis_b6r_vs_b4v2.json --hf-sync-repo m97j/aw-runs-b6

# Causal reward-shape effect (same parent/data/budget/seed, one knob changed):
!python scripts/run_analysis.py \
  --run-a runs/{B6R_EVAL} --label-a b6r-grpo-gated --run-b runs/{B6_EVAL} --label-b b6-grpo-aggregate \
  --output runs/{B6R_EVAL}/analysis_b6r_vs_b6.json --hf-sync-repo m97j/aw-runs-b6

# Flip-level re-diagnosis vs parent (success signature: pass->fail flips with
# score >= 0.8 sharply reduced vs B6's 40–50% near-miss cluster):
!python scripts/x19_b6_regression_diag.py \
  --run-a runs/{B6R_EVAL} --run-b runs/{B4V2_EVAL} \
  --label-a b6r-grpo-gated --label-b b4v2-sft \
  --out runs/x19_b6r_regression_diag.json

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/577 [00:00<?, ?B/s]           
Reconstructing (incomplete total...):   9% 577/6.67k [00:00<00:02, 2.73kB/s]

Fetching 19 files:   5% 1/19 [00:00<00:03,  4.81it/s]
Reconstructing (incomplete total...):   1% 6.67k/455k [00:00<02:44, 2.73kB/s]
Reconstructing (incomplete total...):   1% 6.67k/461k [00:00<02:46, 2.73kB/s]
Reconstructing (incomplete total...):   1% 12.8k/912k [00:00<05:29, 2.73kB/s]
Reconstructing (incomplete total...):   1% 12.8k/1.36M [00:00<08:13, 2.73kB/s]
Reconstructing (incomplete total...):   1% 12.8k/1.74M [00:00<10:34, 2.73kB/s]
Reconstructing (incomplete total...):  80% 1.74M/2.18M [00:00<02:39, 2.73kB/s]
Reconstructing (incomplete total...): 100% 2.18M/2.18M [00:00<00:00, 2.73kB/s]
Reconstructing (incomplete total...):  83% 2.18M/2.63M [00:00<00:00, 5.98MB/s]
Reconstructing (incomplete total..

## B6-R checklist
- [x] prerequisites: cells 1–4 rerun on this runtime; x17 verdict PASS
- [x] b_b6r_train: reward/mean starts ≈0.05–0.15 and climbs (scale NOT comparable
      to B6); frac_reward_zero_std declining; entropy > ~0.005; no RewardHealthError;
      lineage verified (parent sha d4fcacdd)
- [x] c_b6r_eval per-suite pass_rate; x09jr truncation/runaway ≈ 0
- [x] f_b6r_analysis: B6-R vs B4v2 (headline) + B6-R vs B6 (reward-shape effect) recorded
- [x] x19 on B6-R: score≥0.8 pass->fail flips sharply reduced vs B6
- [x] §6 decision per the pre-registered rule → final Track-B champion → report